# What masses fit MoM-BH\*-1? AI-assisted inference (stage 4)

**The question:** stages 2 and 3 used fixed guesses for the uncertain numbers (the efficiency η and the brightness factor f) and tried just a few values. Here every uncertain number varies at once, and we ask: **which combinations of black-hole mass, total mass, η and f reproduce your measured luminosity and temperature?** The spread of the combinations that fit gives the black-hole mass with honest error bars.

**The AI part, in plain words:**
1. Run the stage 2 simulator 60,000 times with random inputs.
2. Train a **neural network** to imitate it: given the inputs, predict the surface temperature.
3. Push **2 million random combinations** through the network and give each a score for how well it matches your measurements. The high-scoring ones are your answer.

**An honest note for your write-up:** the network doesn't add any physics; it only imitates the simulator. Your simulator is already fast, so here the network doesn't save much time. But this is exactly how researchers handle slow simulators (MESA-QUEST takes hours per run), and the same notebook would work if you swapped one in. Because the network is only an imitation, this notebook **checks it against the real simulator twice** (Steps 5 and 7). Those checks are what make the AI part publishable.

**How to use:** run the grey code cells from top to bottom with **Shift + Enter**. Training the network takes about a minute; everything else takes seconds.

## Step 1 — Set up: constants, your measurements and the ranges to explore

**Your measurements, with uncertainties:**
- **Luminosity:** 2.9 × 10⁴⁴ erg/s, ± 0.1 in log (about ±25%). That covers your size notebook (2.5 × 10⁴⁴) and the paper (3.2 × 10⁴⁴).
- **Temperature:** 5,200 ± 300 K, covering your two values, 4,970 K and 5,500 K.
- **The simulator's own error:** ±10% in temperature, from the comparison with Ball et al. in stage 2, Step 5c.

**The ranges to explore (the "priors").** Every combination inside these ranges is treated as equally likely before looking at your data:
- total mass: 300,000 to 100 million Suns;
- black-hole mass: 30 to 30 million Suns, which **includes the paper's 1–10 million**;
- efficiency η: 0.03 to 0.3;
- brightness factor f: 0.76 to 1.58 (the range Ball et al.'s full model covered).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import minimize_scalar

# --- constants (cgs units) ---
G, c, sigma_sb = 6.674e-8, 2.998e10, 5.6704e-5
Msun = 1.989e33
X = 0.7
KAPPA_ES = 0.2 * (1 + X)          # 0.34 cm^2/g
GAMMA, n = 4/3, 3

# --- your measurements and their uncertainties ---
LOGL_OBS, SIG_LOGL = np.log10(2.9e44), 0.1          # luminosity (log10 erg/s), +/- 0.1 in log
T_OBS, SIG_T = 5200, 300                             # temperature (K)
MODEL_ERR = 0.10                                     # simulator's own temperature error (stage 2, Step 5c)
SIG_LOGT = np.sqrt((SIG_T / T_OBS / np.log(10))**2 + np.log10(1 + MODEL_ERR)**2)

# --- ranges to explore (log10 of Suns for masses) ---
PRIOR = {"log_total": (5.5, 8.0), "log_bh": (1.5, 7.5),
         "log_eta": (np.log10(0.03), np.log10(0.3)), "f": (0.76, 1.58)}
rng = np.random.default_rng(42)                      # fixed seed: you get the same numbers every run

def draw(N):
    # N random combinations, spread evenly over the ranges
    return np.column_stack([rng.uniform(*PRIOR[k], N) for k in PRIOR])

print(f"Temperature uncertainty used: +/- {SIG_LOGT:.3f} in log (about {100*(10**SIG_LOGT-1):.0f}%)")

# --- plot style and colours (same as stages 2-3) ---
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": "#0b0b0b", "axes.titlecolor": "#0b0b0b", "xtick.color": "#52514e",
    "ytick.color": "#52514e", "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
    "font.size": 11, "legend.frameon": False, "legend.fontsize": 9.5})
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK2, MUTED, ZONE = "#52514e", "#898781", "#ebeae5"

## Step 2 — The simulator (stage 2, sped up)

This is the same physics as stage 2, rewritten so it handles thousands of combinations in one go. It pre-computes the 300 envelope shapes (a few seconds), then predicts the luminosity, L = f × L_Edd(total mass), and the surface temperature for any combination. Combinations where the black hole is beyond the 10% limit are marked invalid.

**Check:** with stage 2's example (2 million Suns, 10,000-Sun black hole, η = 0.1, your luminosity), it must give **4,947 K**, exactly as before.

In [ ]:
def lane_emden(xi, y):
    theta, phi = y
    return [-phi / xi**2, xi**2 * max(theta, 0.0)**n]

def at_surface(xi, y):
    return y[0]
at_surface.terminal, at_surface.direction = True, -1

def loaded_polytrope(xi0):
    phi_bh = xi0 / (2*n)
    sol = solve_ivp(lane_emden, [xi0, 100], [1.0, phi_bh + (2/3)*xi0**3], events=at_surface,
                    rtol=1e-9, atol=1e-12)
    return phi_bh, sol.y_events[0][0][1], sol.t_events[0][0]       # phi_bh, phi_surface, xi1

best = minimize_scalar(lambda x: -loaded_polytrope(x)[0]/loaded_polytrope(x)[1], bounds=(0.8, 1.2),
                       method="bounded", options={"xatol": 1e-7})
XI0_MAX, Q_MAX = best.x, -best.fun
_xi0 = np.geomspace(1e-7, XI0_MAX, 300)
_tab = np.array([loaded_polytrope(x) for x in _xi0])
_q, _phi_s, _xi1 = _tab[:, 0]/_tab[:, 1], _tab[:, 1], _tab[:, 2]

def simulate(log_total, log_bh, log_eta, f):
    # luminosity (log10 erg/s) and surface temperature (log10 K) for arrays of combinations
    M_tot, M_bh, eta = 10**log_total*Msun, 10**log_bh*Msun, 10**log_eta
    q = M_bh / M_tot
    valid = q < Q_MAX
    lq = np.log(np.clip(q, 1e-12, 0.999999*Q_MAX))
    xi0 = np.exp(np.interp(lq, np.log(_q), np.log(_xi0)))
    phi_s = np.exp(np.interp(lq, np.log(_q), np.log(_phi_s)))
    xi1 = np.interp(lq, np.log(_q), _xi1)
    K = np.pi*G * (M_tot / (4*np.pi*phi_s))**(2/3)
    L = f * 4*np.pi*G*c*M_tot / KAPPA_ES
    A = (16*np.pi*eta/GAMMA) * (G*M_bh)**2 / np.sqrt(GAMMA*K)
    R = xi1 * np.sqrt(K/(np.pi*G)) * (L/A)**(-2/5)
    T = (L / (4*np.pi*sigma_sb*R**2))**0.25
    return np.log10(L), np.where(valid, np.log10(T), np.nan), valid

f_example = 2.9e44 / (4*np.pi*G*c*2e6*Msun/KAPPA_ES)        # the f that gives your luminosity at 2e6 Suns
_, logT_ex, _ = simulate(np.log10(2e6), 4.0, -1.0, f_example)
print(f"Black-hole limit: {Q_MAX:.4f}")
print(f"Stage 2 example: {10**logT_ex:,.0f} K (stage 2 gave 4,947 K)")

## Step 3 — Make the training data

Run the simulator for 60,000 random combinations from the ranges in Step 1, and keep the valid ones (black hole below the limit). These are the examples the network will learn from.

The network gets four inputs: log total mass, **log black-hole share** (black hole ÷ total), log η and f. The share is used instead of the black-hole mass because the physics depends on it directly, which makes it easier to learn. The output is log surface temperature. Luminosity doesn't need learning: it's simply f × L_Edd.

In [ ]:
def features(P):
    # network inputs: log total, log black-hole share, log eta, f
    return np.column_stack([P[:, 0], P[:, 1] - P[:, 0], P[:, 2], P[:, 3]])

P_train = draw(60000)
_, logT_train, valid = simulate(*P_train.T)
P_train, logT_train = P_train[valid], logT_train[valid]
print(f"{len(P_train):,} valid examples out of 60,000")
print(f"Temperatures in the training data: {10**logT_train.min():,.0f} - {10**logT_train.max():,.0f} K")

## Step 4 — Train the neural network

The network is a **multilayer perceptron**: three layers of 64 "neurons" each, a standard design for learning a smooth function. scikit-learn (already in Colab) trains it.

The examples are split 80% for training and 20% for **testing**. The network never sees the test examples while learning, so Step 5 can check how well it does on combinations it hasn't met. Training stops automatically once it stops improving.

**This takes about a minute.**

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import time

X_all = features(P_train)
X_tr, X_te, y_tr, y_te = train_test_split(X_all, logT_train, test_size=0.2, random_state=0)
scaler = StandardScaler().fit(X_tr)            # puts every input on a similar scale

start = time.time()
net = MLPRegressor(hidden_layer_sizes=(64, 64, 64), activation="tanh", learning_rate_init=3e-3,
                   max_iter=600, early_stopping=True, n_iter_no_change=30, tol=1e-7, random_state=0)
net.fit(scaler.transform(X_tr), y_tr)
print(f"Trained on {len(y_tr):,} examples in {time.time()-start:.0f} s ({net.n_iter_} rounds)")

def net_logT(P):
    return net.predict(scaler.transform(features(P)))

## Step 5 — AI check 1: how accurate is the network?

Compare the network's temperature with the real simulator's for the 20% of examples it never saw. Two summaries:
- **everywhere** in the ranges;
- **where it matters:** temperatures of 3,500–8,000 K with the black hole under 5% of the mass, the region around MoM-BH\*-1.

**PASS** if 99% of the "where it matters" cases are within 2%. That's far smaller than your ~11% temperature uncertainty, so the network's error can't change the answer.

The plot shows where the errors are. They're largest right at the black-hole limit, where the simulator's temperature changes abruptly (the upturn in stage 2, Step 7).

In [ ]:
T_true = 10**y_te
T_pred = 10**net.predict(scaler.transform(X_te))
err = 100 * (T_pred / T_true - 1)                        # percent
share = 10**X_te[:, 1]
matters = (T_true > 3500) & (T_true < 8000) & (share < 0.05)

print(f"Everywhere ({len(err):,} tests):   typical error {np.median(np.abs(err)):.2f}%, "
      f"99% within {np.percentile(np.abs(err), 99):.2f}%, worst {np.abs(err).max():.1f}%")
print(f"Where it matters ({matters.sum():,} tests): typical error {np.median(np.abs(err[matters])):.2f}%, "
      f"99% within {np.percentile(np.abs(err[matters]), 99):.2f}%, worst {np.abs(err[matters]).max():.1f}%")
ok = np.percentile(np.abs(err[matters]), 99) < 2
print(f"{'PASS' if ok else 'FAIL'}  network accuracy where it matters")

example = np.array([[np.log10(2e6), 4.0, -1.0, f_example]])
print(f"Stage 2 example: network {10**net_logT(example)[0]:,.0f} K, simulator {10**logT_ex:,.0f} K")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
ax1.plot(T_true[~matters], T_pred[~matters], ".", color=MUTED, ms=3, alpha=0.5, label="elsewhere")
ax1.plot(T_true[matters], T_pred[matters], ".", color=BLUE, ms=3, alpha=0.6, label="where it matters")
ax1.plot([800, 3e5], [800, 3e5], color=INK2, lw=1)
ax1.set_xscale("log"); ax1.set_yscale("log")
ax1.set_xlabel("Simulator temperature (K)"); ax1.set_ylabel("Network temperature (K)")
ax1.set_title("Network vs simulator (unseen tests)", loc="left")
ax1.legend(loc="upper left", markerscale=4)
ax2.plot(share[~matters], np.abs(err[~matters]), ".", color=MUTED, ms=3, alpha=0.5)
ax2.plot(share[matters], np.abs(err[matters]), ".", color=BLUE, ms=3, alpha=0.6)
ax2.axhline(2, color=INK2, lw=1, ls="--")
ax2.text(1.5e-6, 2.4, "2% pass line", color=INK2, fontsize=9.5)
ax2.axvline(Q_MAX, color=MUTED, lw=1, ls=":")
ax2.text(Q_MAX*0.9, 12, "black-hole\nlimit", color=INK2, fontsize=9.5, ha="right")
ax2.set_xscale("log"); ax2.set_yscale("log")
ax2.set_ylim(1e-4, 30)
ax2.set_xlabel("Black hole's share of the total mass"); ax2.set_ylabel("Network error (%)")
ax2.set_title("Errors are largest only near the limit", loc="left")
plt.tight_layout(); fig.savefig("s4_step5_network_check.png", dpi=150); plt.show()

## Step 6 — The inference: which combinations fit your measurements?

Draw **2 million random combinations** from the ranges and use the network to predict each one's temperature. Then give each a **score** for how well it matches your measurements: 1 for a perfect match, falling off the further its luminosity and temperature are from yours (relative to the uncertainties). This is Bayesian inference by **importance sampling**.

The answer is the score-weighted spread of the inputs: the middle value (median) and the range holding the middle 68% (like a ±1σ error bar).

**Two versions,** because stage 2 found the simulator runs cooler than Ball et al.'s full model:
- **as is:** the simulator's temperature with its ±10% uncertainty;
- **calibrated:** each temperature corrected by the stage 2, Step 5c ratio for its black-hole share, before scoring.

The **combined** answer gives both versions equal weight. Use it as your headline, because it includes this systematic uncertainty.

In [ ]:
CAL_SHARE = np.array([5e-4, 0.01, 0.02, 0.05, 0.1])          # stage 2, Step 5c: model / Ball temperature
CAL_RATIO = np.array([1.10, 0.83, 0.73, 0.61, 0.68])

def infer(N, use_network=True, calibrate=False):
    P = draw(N)
    logL, logT_sim, valid = simulate(*P.T)
    logT = np.full(N, np.nan)
    logT[valid] = net_logT(P[valid]) if use_network else logT_sim[valid]
    if calibrate:
        ratio = np.interp(P[:, 1] - P[:, 0], np.log10(CAL_SHARE), CAL_RATIO)
        logT = logT - np.log10(ratio)
    chi2 = ((logL - LOGL_OBS)/SIG_LOGL)**2 + ((logT - np.log10(T_OBS))/SIG_LOGT)**2
    score = np.where(valid, np.exp(-0.5*np.nan_to_num(chi2, nan=1e9)), 0.0)
    return P, score

def weighted_range(x, w):
    # median and the middle 68% of x, weighted by the scores
    order = np.argsort(x)
    cum = np.cumsum(w[order]); cum /= cum[-1]
    return np.interp([0.16, 0.5, 0.84], cum, x[order])

def report(P, w, label):
    rows = [("total mass (Suns)", 0, True, "{:.2e}"), ("black hole (Suns)", 1, True, "{:,.0f}"),
            ("efficiency eta", 2, True, "{:.3f}"), ("brightness factor f", 3, False, "{:.2f}")]
    print(f"--- {label} (effective number of good matches: {w.sum()**2/(w**2).sum():,.0f}) ---")
    for name, i, is_log, fmt in rows:
        lo, mid, hi = weighted_range(P[:, i], w)
        if is_log:
            lo, mid, hi = 10**lo, 10**mid, 10**hi
        print(f"   {name:22s} {fmt.format(mid):>9s}   (68% range {fmt.format(lo)} - {fmt.format(hi)})")
    lo, mid, hi = 10**weighted_range(P[:, 1] - P[:, 0], w)
    print(f"   {'black-hole share':22s} {mid:9.4f}   (68% range {lo:.4f} - {hi:.4f})")
    p_big = w[P[:, 1] > 6].sum() / w.sum()
    print("   chance the black hole is over 1 million Suns: "
          + (f"{p_big:.1e}" if p_big >= 1e-6 else "less than one in a million"))

start = time.time()
P_raw, w_raw = infer(2_000_000)
P_cal, w_cal = infer(2_000_000, calibrate=True)
print(f"(4 million combinations scored in {time.time()-start:.0f} s)\n")
report(P_raw, w_raw, "simulator as is")
report(P_cal, w_cal, "calibrated to Ball et al.")

# combined: give each version equal total weight
P_all = np.vstack([P_raw, P_cal])
w_all = np.concatenate([w_raw / w_raw.sum(), w_cal / w_cal.sum()])
report(P_all, w_all, "COMBINED (headline)")

**Reading the numbers:**
- **The black hole** comes out at around 10,000 Suns. The combined 68% range runs from a few thousand to a few tens of thousands of Suns.
- **The total mass** is about 1.3–2.4 million Suns, set mainly by your luminosity and the Eddington limit.
- **η and f** come out spread across their whole ranges. Your two measurements can't pin them down, and that's fine: their uncertainty is now *inside* the error bars instead of being guessed.
- **The paper's 1–10 million Suns** gets a chance of essentially zero, but only *within this model*, because such a black hole can't sit in a balanced envelope with your luminosity.

## Step 7 — AI check 2: does the network change the answer?

Repeat the "as is" inference, but this time use the **real simulator** instead of the network, for 400,000 combinations. If the network imitates the simulator well, both give the same answer. **PASS** if the black hole's median and 68% range agree within 5%.

In [ ]:
P_sim, w_sim = infer(400_000, use_network=False)
lo_n, mid_n, hi_n = 10**weighted_range(P_raw[:, 1], w_raw)
lo_s, mid_s, hi_s = 10**weighted_range(P_sim[:, 1], w_sim)
print(f"Black hole with the network:   {mid_n:,.0f} (68% range {lo_n:,.0f} - {hi_n:,.0f})")
print(f"Black hole with the simulator: {mid_s:,.0f} (68% range {lo_s:,.0f} - {hi_s:,.0f})")
worst = max(abs(mid_n/mid_s - 1), abs(lo_n/lo_s - 1), abs(hi_n/hi_s - 1))
print(f"{'PASS' if worst < 0.05 else 'FAIL'}  network and simulator agree within {100*worst:.1f}%")

## Step 8 — Plot the answer

**Left:** how likely each black-hole mass is, for the two versions and combined. The orange band is the paper's line-width estimate.
**Right:** 5,000 combinations picked according to their scores, with total mass across and black-hole mass up. The grey zone is where no balanced envelope can exist.

In [ ]:
bins = np.linspace(1.5, 7.5, 121)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8))
ax1.axvspan(1e6, 1e7, color=ORANGE, alpha=0.2, lw=0, label="paper's estimate (line widths)")
peak = 0
for P, w, col, lab in [(P_raw, w_raw, BLUE, "simulator as is"), (P_cal, w_cal, AQUA, "calibrated to Ball et al.")]:
    h, edges = np.histogram(P[:, 1], bins=bins, weights=w, density=True)
    ax1.step(10**edges[:-1], h, where="post", color=col, lw=1.6, label=lab)
    peak = max(peak, h.max())
h, edges = np.histogram(P_all[:, 1], bins=bins, weights=w_all, density=True)
ax1.fill_between(10**edges[:-1], h, step="post", color=INK2, alpha=0.15, lw=0, label="combined")
ax1.set_xscale("log"); ax1.set_xlim(1e2, 3e7); ax1.set_ylim(0, 1.7*peak)     # headroom for the legend
ax1.set_xlabel("Black-hole mass (Suns)"); ax1.set_ylabel("How likely (per log Suns)")
ax1.set_title("Your measurements point to ~10⁴ Suns", loc="left")
ax1.legend(loc="upper left")

pick = rng.choice(len(w_all), size=5000, p=w_all/w_all.sum())
x = np.logspace(5, 8.5, 50)
ax2.fill_between(x, Q_MAX*x, 1e9, color=ZONE, zorder=0, lw=0)
ax2.text(4e5, 3e7, "no balanced envelope", color=INK2, fontsize=10)
ax2.axhspan(1e6, 1e7, color=ORANGE, alpha=0.2, lw=0)
ax2.plot(10**P_all[pick, 0], 10**P_all[pick, 1], ".", color=BLUE, ms=3, alpha=0.4)
ax2.set_xscale("log"); ax2.set_yscale("log")
ax2.set_xlim(3e5, 1e8); ax2.set_ylim(1e2, 1e8)
ax2.set_xlabel("Total mass (Suns)"); ax2.set_ylabel("Black-hole mass (Suns)")
ax2.set_title("Combinations that fit (combined)", loc="left")
plt.tight_layout(); fig.savefig("s4_step8_posterior.png", dpi=150); plt.show()

## Step 9 — Into stage 3: how far through its life?

In stage 3, the black hole gains the same amount of mass every year. So **the share of its life used so far equals its black-hole share divided by the limit** (q ÷ 0.1015). This doesn't depend on η, f or the efficiency ε. Its **age** does: it's that share × the lifetime, 4.34 Myr ÷ f, for ε = 0.1.

In [ ]:
life_used = 10**(P_all[:, 1] - P_all[:, 0]) / Q_MAX
age_myr = life_used * 4.34 / P_all[:, 3]
lo, mid, hi = weighted_range(life_used, w_all)
print(f"Share of its life used so far: {100*mid:.1f}%  (68% range {100*lo:.1f} - {100*hi:.1f}%)")
lo, mid, hi = weighted_range(age_myr, w_all)
print(f"Age (efficiency 0.1):          {mid:.2f} million years  (68% range {lo:.2f} - {hi:.2f})")

## Step 10 — Check against an independent measurement

Curtis et al. (2026) measured MoM-BH\*-1's **surface gravity** by fitting the shape of its spectrum. In stage 2, Step 12, that gave a total mass of about **2.4 million Suns** (range 0.35–6.9 million). Your inference never used it, so it's a fair test: does your total mass agree?

In [ ]:
lo, mid, hi = 10**weighted_range(P_all[:, 0], w_all)
print(f"Your inferred total mass:          {mid:.2e} Suns (68% range {lo:.2e} - {hi:.2e})")
print(f"Curtis et al.'s gravity-based mass: 2.4e+06 Suns (range 3.5e+05 - 6.9e+06)")
overlap = (lo < 6.9e6) and (hi > 3.5e5)
print(f"{'PASS' if overlap else 'FAIL'}  the two ranges {'overlap' if overlap else 'do not overlap'}")

# What stage 4 tells you

- **The black hole:** about 10,000 Suns (combined 68% range: a few thousand to a few tens of thousands; see Step 6 for your exact numbers). That range now includes the uncertainties in the luminosity, the temperature, the efficiency η, the brightness factor f and the simulator's own error.
- **The total mass:** about 1.3–2.4 million Suns, consistent with Curtis et al.'s independent surface-gravity mass (Step 10).
- **MoM-BH\*-1 is early in its life**, in the first few percent to roughly 20%, at an age of a few hundred thousand years (Step 9).
- **A classic quasi-star can't hold the paper's 1–10 million Suns** with this luminosity: the chance comes out less than one in a million (Step 6).

**The AI checks:** the network is accurate to better than 1% where it matters (Step 5). Swapping it for the real simulator changes the answer by only a few percent (Step 7).

**A method sentence for your note** (fill in your numbers from Steps 4–7):
*"We emulate the semi-analytic quasi-star model with a multilayer perceptron (three hidden layers of 64 units), trained on ~34,000 simulator runs drawn from the priors. On held-out runs, the network reproduces the effective temperature to within X% (99th percentile) for T = 3,500–8,000 K and black-hole share below 0.05. Posteriors are computed by importance sampling of 2 × 10⁶ prior draws and agree with direct-simulator inference to within Y%."*

**What the AI does not change:** everything still rests on the classic quasi-star assumption and on the stage 2 physics. The network inherits the simulator's limitations; it doesn't remove them.